In [1]:
# confirm in our virtual environment
import sys
print(sys.executable)

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/bin/python


In [2]:
import os 
print(os.getcwd())

/Users/hussaintaheri/Desktop/sports-win-predictor/notebooks


In [3]:
import pandas as pd
df = pd.read_csv("../data/nba_pbp_2020_2025.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0,0,22401186,2,PT12M00.00S,1,0,NaN,0,NaN,...,0.0,0.0,0,NaN,Start of 1st Period (1:12 PM EST),period,start,0,0,1
1,1,1,22401186,4,PT12M00.00S,1,1610612737,ATL,1631230,Barlow,...,NaN,NaN,0,h,Jump Ball Barlow vs. Carter Jr.: Tip to Mann,Jump Ball,NaN,1,0,2
2,2,2,22401186,7,PT11M43.00S,1,1610612737,ATL,1629611,Mann,...,NaN,NaN,0,h,MISS Mann 4' Fadeaway Jumper,Missed Shot,Fadeaway Jump Shot,1,2,3
3,3,3,22401186,8,PT11M40.00S,1,1610612753,ORL,1628976,Carter Jr.,...,NaN,NaN,0,v,Carter Jr. REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,4
4,4,4,22401186,9,PT11M23.00S,1,1610612753,ORL,203484,Caldwell-Pope,...,NaN,NaN,0,v,MISS Caldwell-Pope 25' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,5


In [4]:
# view columns to drop which ones we want be using for our model
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'gameId', 'actionNumber', 'clock',
       'period', 'teamId', 'teamTricode', 'personId', 'playerName',
       'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult',
       'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location',
       'description', 'actionType', 'subType', 'videoAvailable', 'shotValue',
       'actionId'],
      dtype='object')

In [6]:
df.isna().count()

gameId        3282178
clock         3282178
period        3282178
scoreHome     3282178
scoreAway     3282178
location      3282178
actionType    3282178
dtype: int64

In [7]:
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period
2,22401186,PT11M43.00S,1,NaN,NaN,h,Missed Shot
3,22401186,PT11M40.00S,1,NaN,NaN,v,Rebound
4,22401186,PT11M23.00S,1,NaN,NaN,v,Missed Shot
5,22401186,PT11M19.00S,1,NaN,NaN,h,Rebound
6,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot
7,22401186,PT10M58.00S,1,NaN,NaN,v,Missed Shot
8,22401186,PT10M55.00S,1,NaN,NaN,v,Rebound
9,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot
10,22401186,PT10M35.00S,1,NaN,NaN,h,Turnover


In [8]:
# fill missing values for home and away so it carries the values forward and not update whenever a score changes
# use forward fill method
df[["scoreHome","scoreAway"]] = df.groupby(["gameId"])[["scoreHome", "scoreAway"]].ffill()

In [9]:
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period
2,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot
3,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound
4,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot
5,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound
6,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot
7,22401186,PT10M58.00S,1,2.0,0.0,v,Missed Shot
8,22401186,PT10M55.00S,1,2.0,0.0,v,Rebound
9,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot
10,22401186,PT10M35.00S,1,2.0,3.0,h,Turnover


In [10]:
# get the last row of every game
temp_df = df.groupby("gameId").last().reset_index()

# create target column for both dataframes comparing to see if home team won or not
temp_df["home_team_won"] = (temp_df["scoreHome"] > temp_df["scoreAway"]).astype(int)

# now merge both dataframes so we know the target value for all rows
df = pd.merge(df, temp_df[["home_team_won", "gameId"]], on = "gameId")

In [11]:
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period,1
1,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot,1
2,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound,1
3,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot,1
4,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound,1
5,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot,1
6,22401186,PT10M58.00S,1,2.0,0.0,v,Missed Shot,1
7,22401186,PT10M55.00S,1,2.0,0.0,v,Rebound,1
8,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot,1
9,22401186,PT10M35.00S,1,2.0,3.0,h,Turnover,1


In [12]:
df.tail(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won
3282158,22000496,PT00M06.90S,4,119.0,122.0,h,Rebound,0
3282159,22000496,PT00M06.90S,4,119.0,122.0,v,Foul,0
3282160,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0
3282161,22000496,PT00M06.90S,4,120.0,122.0,v,Substitution,0
3282162,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0
3282163,22000496,PT00M05.80S,4,120.0,122.0,v,Rebound,0
3282164,22000496,PT00M05.80S,4,120.0,122.0,h,Foul,0
3282165,22000496,PT00M05.80S,4,120.0,123.0,v,Free Throw,0
3282166,22000496,PT00M05.80S,4,120.0,123.0,h,Substitution,0
3282167,22000496,PT00M05.80S,4,120.0,124.0,v,Free Throw,0


In [13]:
df.dtypes

gameId             int64
clock             object
period             int64
scoreHome        float64
scoreAway        float64
location          object
actionType        object
home_team_won      int64
dtype: object

In [14]:
# check how much periods in our df
df["period"].value_counts()

period
4    832755
2    830947
3    804796
1    791902
5     19474
6      2076
7       228
Name: count, dtype: int64

In [15]:
df.head()

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period,1
1,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot,1
2,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound,1
3,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot,1
4,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound,1


In [16]:
# created a function to convert clock and period columns to a single column of just seconds
def parse_clock(clock, period):
    clock = clock.split("M")
    clock[0] = clock[0][2:]
    clock[1] = clock[1][:-4]

    clock_seconds_left = int(clock[0]) * 60 + int(clock[1])

    if period < 5:
        seconds_before_period = (period - 1) * 720
        seconds = seconds_before_period + (720 - clock_seconds_left)
    else:
        seconds_before_period = 2880 + (period - 5) * 300
        seconds = seconds_before_period + (300 - clock_seconds_left)

    return seconds

In [17]:
# apply the function to the data frame
df["seconds"] = df.apply(lambda row: parse_clock(row["clock"], row["period"]), axis = 1)
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won,seconds
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period,1,0
1,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot,1,17
2,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound,1,20
3,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot,1,37
4,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound,1,41
5,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot,1,45
6,22401186,PT10M58.00S,1,2.0,0.0,v,Missed Shot,1,62
7,22401186,PT10M55.00S,1,2.0,0.0,v,Rebound,1,65
8,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot,1,71
9,22401186,PT10M35.00S,1,2.0,3.0,h,Turnover,1,85


In [18]:
df.tail(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won,seconds
3282158,22000496,PT00M06.90S,4,119.0,122.0,h,Rebound,0,2874
3282159,22000496,PT00M06.90S,4,119.0,122.0,v,Foul,0,2874
3282160,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0,2874
3282161,22000496,PT00M06.90S,4,120.0,122.0,v,Substitution,0,2874
3282162,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0,2874
3282163,22000496,PT00M05.80S,4,120.0,122.0,v,Rebound,0,2875
3282164,22000496,PT00M05.80S,4,120.0,122.0,h,Foul,0,2875
3282165,22000496,PT00M05.80S,4,120.0,123.0,v,Free Throw,0,2875
3282166,22000496,PT00M05.80S,4,120.0,123.0,h,Substitution,0,2875
3282167,22000496,PT00M05.80S,4,120.0,124.0,v,Free Throw,0,2875


In [19]:
# count the amount of values for seconds
df["seconds"].isna().sum()

np.int64(0)

In [20]:
# now drop the clock column
df = df.drop(["clock"], axis = 1)
df.head()

,gameId,period,scoreHome,scoreAway,location,actionType,home_team_won,seconds
0,22401186,1,0.0,0.0,NaN,period,1,0
1,22401186,1,0.0,0.0,h,Missed Shot,1,17
2,22401186,1,0.0,0.0,v,Rebound,1,20
3,22401186,1,0.0,0.0,v,Missed Shot,1,37
4,22401186,1,0.0,0.0,h,Rebound,1,41


In [21]:
df.dtypes

gameId             int64
period             int64
scoreHome        float64
scoreAway        float64
location          object
actionType        object
home_team_won      int64
seconds            int64
dtype: object

In [22]:
# check the amount of different values in actiontype
df["actionType"].value_counts()

actionType
Rebound           698025
Missed Shot       632259
Made Shot         555059
Substitution      314165
Free Throw        296706
Foul              266823
Turnover          186887
Timeout            73560
period             54470
Instant Replay     12557
Jump Ball          11786
Violation          11652
Ejection             482
Name: count, dtype: int64

In [23]:
# create 13 new columns and a 1 for what actiontype it is and the rest 0s to have our model to be able to know the importance of each type
df = pd.get_dummies(df, columns = ["actionType"], dtype = int)
df.head()

,gameId,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,22401186,1,0.0,0.0,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,22401186,1,0.0,0.0,h,1,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,22401186,1,0.0,0.0,v,1,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,22401186,1,0.0,0.0,v,1,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,22401186,1,0.0,0.0,h,1,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [24]:
df.dtypes

gameId                         int64
period                         int64
scoreHome                    float64
scoreAway                    float64
location                      object
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object

In [25]:
df["location"].value_counts()

location
h    1616252
v    1597585
Name: count, dtype: int64

In [26]:
df["location"].isna().sum()

np.int64(68341)

In [27]:
# replace h with 2 and v with 1 and 0 for NaN values for locatin column in the df
df["location"] = df["location"].map({"h": 2, "v": 1})
df["location"] = df["location"].fillna(0)
df.head()

,gameId,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,22401186,1,0.0,0.0,0.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,22401186,1,0.0,0.0,2.0,1,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,22401186,1,0.0,0.0,1.0,1,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,22401186,1,0.0,0.0,1.0,1,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,22401186,1,0.0,0.0,2.0,1,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [28]:
df.dtypes

gameId                         int64
period                         int64
scoreHome                    float64
scoreAway                    float64
location                     float64
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object

In [29]:
df.isna().sum()

gameId                       0
period                       0
scoreHome                    1
scoreAway                    1
location                     0
home_team_won                0
seconds                      0
actionType_Ejection          0
actionType_Foul              0
actionType_Free Throw        0
actionType_Instant Replay    0
actionType_Jump Ball         0
actionType_Made Shot         0
actionType_Missed Shot       0
actionType_Rebound           0
actionType_Substitution      0
actionType_Timeout           0
actionType_Turnover          0
actionType_Violation         0
actionType_period            0
dtype: int64

In [30]:
# check which row in the dataframe has the null values
df[df["scoreHome"].isna()]

,gameId,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
2058904,22100880,1,NaN,NaN,0.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [31]:
df[2058904:2058914]

,gameId,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
2058904,22100880,1,NaN,NaN,0.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2058905,22100880,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2058906,22100880,1,0.0,0.0,1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2058907,22100880,1,0.0,0.0,2.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2058908,22100880,1,0.0,0.0,2.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2058909,22100880,1,0.0,0.0,1.0,0,10,0,0,0,0,0,0,0,0,0,0,1,0,0
2058910,22100880,1,0.0,0.0,2.0,0,34,0,0,0,0,0,0,0,0,0,0,1,0,0
2058911,22100880,1,0.0,2.0,1.0,0,44,0,0,0,0,0,1,0,0,0,0,0,0,0
2058912,22100880,1,0.0,2.0,2.0,0,61,0,0,0,0,0,0,1,0,0,0,0,0,0
2058913,22100880,1,0.0,2.0,1.0,0,63,0,0,0,0,0,0,0,1,0,0,0,0,0


In [32]:
# drop the row to have that has null values to finally have a dataset where our model can train on
df = df.drop(2058904)

In [33]:
df.dtypes

gameId                         int64
period                         int64
scoreHome                    float64
scoreAway                    float64
location                     float64
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object

In [34]:
df.isna().sum()

gameId                       0
period                       0
scoreHome                    0
scoreAway                    0
location                     0
home_team_won                0
seconds                      0
actionType_Ejection          0
actionType_Foul              0
actionType_Free Throw        0
actionType_Instant Replay    0
actionType_Jump Ball         0
actionType_Made Shot         0
actionType_Missed Shot       0
actionType_Rebound           0
actionType_Substitution      0
actionType_Timeout           0
actionType_Turnover          0
actionType_Violation         0
actionType_period            0
dtype: int64

In [35]:
# drop the gameId since its just an identifier and our model cant learn from it
df = df.drop(columns = ["gameId"], axis = 1)

In [36]:
df.head()

,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,1,0.0,0.0,0.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,0.0,0.0,2.0,1,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,1,0.0,0.0,1.0,1,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,1,0.0,0.0,1.0,1,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,1,0.0,0.0,2.0,1,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [37]:
# lets create the features and target
X = df.drop(columns = ["home_team_won"])
y = df["home_team_won"]

In [38]:
X.head()

,period,scoreHome,scoreAway,location,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,0.0,0.0,2.0,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,1,0.0,0.0,1.0,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,1,0.0,0.0,1.0,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,1,0.0,0.0,2.0,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [39]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: home_team_won, dtype: int64

In [40]:
# we found 2625533 to be the start of a new game which is really close to 80 percent of the data where we will split at
X[2625530:]

,period,scoreHome,scoreAway,location,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
2625531,4,102.0,110.0,1.0,2880,0,0,0,0,0,0,0,1,0,0,0,0,0
2625532,4,102.0,110.0,0.0,2880,0,0,0,0,0,0,0,0,0,0,0,0,1
2625533,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2625534,1,0.0,0.0,2.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2625535,1,0.0,0.0,1.0,6,0,0,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3282173,4,121.0,124.0,2.0,2879,0,0,0,0,0,0,0,0,1,0,0,0,0
3282174,4,121.0,124.0,2.0,2879,0,0,0,0,0,0,0,0,1,0,0,0,0
3282175,4,121.0,124.0,2.0,2880,0,0,0,0,0,0,1,0,0,0,0,0,0
3282176,4,121.0,124.0,2.0,2880,0,0,0,0,0,0,0,1,0,0,0,0,0


In [41]:
# since we have time series data will be creating train and test datasets manually
X_train, X_test, y_train, y_test = X.iloc[:2625532], X.iloc[2625532:], y.iloc[:2625532], y.iloc[2625532:]

In [42]:
X_train.tail()

,period,scoreHome,scoreAway,location,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
2625528,4,102.0,110.0,1.0,2875,0,0,0,0,0,0,0,0,0,0,0,0,0
2625529,4,102.0,110.0,2.0,2879,0,0,0,0,0,0,0,1,0,0,0,0,0
2625530,4,102.0,110.0,2.0,2880,0,0,0,0,0,0,1,0,0,0,0,0,0
2625531,4,102.0,110.0,1.0,2880,0,0,0,0,0,0,0,1,0,0,0,0,0
2625532,4,102.0,110.0,0.0,2880,0,0,0,0,0,0,0,0,0,0,0,0,1


In [43]:
X_test.head()

,period,scoreHome,scoreAway,location,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
2625533,1,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2625534,1,0.0,0.0,2.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2625535,1,0.0,0.0,1.0,6,0,0,0,0,0,0,1,0,0,0,0,0,0
2625536,1,0.0,0.0,2.0,10,0,0,0,0,0,0,0,1,0,0,0,0,0
2625537,1,0.0,0.0,2.0,19,0,0,0,0,0,0,1,0,0,0,0,0,0


In [44]:
# check if the data is the same for train and test
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2625532, 18), (656645, 18), (2625532,), (656645,))

In [47]:
%%time
# import the model we want to first use
from xgboost import XGBClassifier

# import the metrics we wanna use
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

# track the models we want to use on our data set
import mlflow

# start an mlflow run
with mlflow.start_run():
    
    # initialize the model and have verbosity to 1 to see the training process
    model = XGBClassifier(verbosity = 1)
    
    # train the model
    xg = model.fit(X_train, y_train)

    # log the parameters in this case the default ones
    mlflow.log_params(model.get_params())

    # log the model and give it a name
    model_info = mlflow.xgboost.log_model(xg, name = "baseline_xgboost_model1")
    
    # get the model predicted values
    y_pred = model.predict(X_test)

    # get the probabilite values, so we can use for log loss and roc_auc
    y_proba = model.predict_proba(X_test)

    # get the models metrics
    score = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba[:, 1])
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])

    # log the metrics
    mlflow.log_metrics({"log_loss": ll, "accuracy_score": score, "roc_auc": roc_auc})

CPU times: user 14 s, sys: 565 ms, total: 14.5 s
Wall time: 6.05 s


In [48]:
%%time
from lightgbm import LGBMClassifier
import mlflow

with mlflow.start_run():
    
    model = LGBMClassifier(verbosity = 1)
    lgb = model.fit(X_train, y_train)
    
    mlflow.log_params(model.get_params())
    model_info = mlflow.lightgbm.log_model(lgb, name = "baseline_lightgbm_model1")
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    score = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba[:, 1])
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])

    mlflow.log_metrics({"log_loss": ll, "accuracy_score": score, "roc_auc": roc_auc})

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1445687, number of negative: 1179845
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004691 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 607
[LightGBM] [Info] Number of data points in the train set: 2625532, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.550626 -> initscore=0.203202
[LightGBM] [Info] Start training from score 0.203202


2026/04/08 19:22:05 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


CPU times: user 15.4 s, sys: 895 ms, total: 16.3 s
Wall time: 6.28 s


In [51]:
%%time
from catboost import CatBoostClassifier
import mlflow

with mlflow.start_run():
    
    model = CatBoostClassifier()
    cb = model.fit(X_train, y_train)
    
    mlflow.log_params(model.get_params())
    model_info = mlflow.catboost.log_model(cb, name = "baseline_catboost_model1")
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    score = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba[:, 1])
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])

    mlflow.log_metrics({"log_loss": ll, "accuracy_score": score, "roc_auc": roc_auc})

Learning rate set to 0.297125
0:	learn: 0.6289533	total: 255ms	remaining: 4m 14s
1:	learn: 0.5972522	total: 365ms	remaining: 3m 2s
2:	learn: 0.5760786	total: 500ms	remaining: 2m 46s
3:	learn: 0.5619212	total: 687ms	remaining: 2m 51s
4:	learn: 0.5479096	total: 818ms	remaining: 2m 42s
5:	learn: 0.5406069	total: 949ms	remaining: 2m 37s
6:	learn: 0.5321447	total: 1.13s	remaining: 2m 40s
7:	learn: 0.5264485	total: 1.3s	remaining: 2m 41s
8:	learn: 0.5220546	total: 1.43s	remaining: 2m 37s
9:	learn: 0.5182182	total: 1.53s	remaining: 2m 31s
10:	learn: 0.5149431	total: 1.67s	remaining: 2m 29s
11:	learn: 0.5127137	total: 1.79s	remaining: 2m 27s
12:	learn: 0.5100572	total: 1.98s	remaining: 2m 30s
13:	learn: 0.5075324	total: 2.09s	remaining: 2m 27s
14:	learn: 0.5062455	total: 2.28s	remaining: 2m 29s
15:	learn: 0.5045828	total: 2.41s	remaining: 2m 28s
16:	learn: 0.5034602	total: 2.58s	remaining: 2m 29s
17:	learn: 0.5027737	total: 2.7s	remaining: 2m 27s
18:	learn: 0.5014066	total: 2.86s	remaining: 2m

In [52]:
# now we want to choose the best hyperparameters for our model and we can automate this by using optuna which also more efficient(bayesian optimization)
import optuna
from xgboost import XGBClassifier
def objective(trial):
    # invoke suggestions of a trial object to generate hyperparameters
    max_depth = trial.suggest_int("max_depth", 3, 10)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 20)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    reg_alpha = trial.suggest_int("reg_alpha", 0, 100)
    reg_lambda = trial.suggest_int("reg_lambda", 0, 10)

    # pass those values into our model
    model = XGBClassifier(max_depth = max_depth, min_child_weight = min_child_weight, subsample = subsample, colsample_bytree = colsample_bytree,
                           learning_rate = learning_rate, n_estimators = n_estimators, reg_alpha = reg_alpha, reg_lambda = reg_lambda)
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # metric we want to optimize
    ll = log_loss(y_test, y_proba[:, 1])
    return ll

# since our main metric is log loss the lower score the better so our goal is to minimize it
study = optuna.create_study(direction = "minimize")

# lets use 1000 trials to find the best hyperparameters we could ever possibly find
study.optimize(objective, n_trials = 1000)

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-08 21:06:19,950] A new study created in memory with name: no-name-3f4253cd-08b8-4463-95e3-af21bee5797c
[I 2026-04-08 21:06:42,542] Trial 0 finished with value: 0.4820435379044586 and parameters: {'max_depth': 6, 'min_child_weight': 18, 'subsample': 0.7710274732458069, 'colsample_bytree': 0.5057084146130187, 'learning_rate': 0.046381924670649846, 'n_estimators': 780, 'reg_alpha': 76, 'reg_lambda': 6}. Best is trial 0 with value: 0.4820435379044586.
[I 2026-04-08 21:06:50,187] Trial 1 finished with value: 0.5130635254026433 and parameters: {'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.8245949475809506, 'colsample_bytree': 0.9022684041602722, 'learning_rate': 0.011607972174178147, 'n_e

In [53]:
# import our feature engineered dataframe back to csv
df.to_csv("../data/nba_pbp_2020_2025_fe.csv")

In [54]:
# find the best parameters
study.best_params

{'max_depth': 6,
 'min_child_weight': 17,
 'subsample': 0.5643733174447721,
 'colsample_bytree': 0.9827635334946612,
 'learning_rate': 0.05072478407219559,
 'n_estimators': 771,
 'reg_alpha': 7,
 'reg_lambda': 10}

In [55]:
study.best_value

0.4801060553554118

In [57]:
%%time
# import the model we want to first use
from xgboost import XGBClassifier

# import the metrics we wanna use
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

# track the models we want to use on our data set
import mlflow

# start an mlflow run
with mlflow.start_run():
    
    # initialize the model and use the best parameters we got and unpack them using **
    model = XGBClassifier(**study.best_params)
    
    # train the model
    xg = model.fit(X_train, y_train)

    # log the parameters in this case the default ones
    mlflow.log_params(model.get_params())

    # log the model and give it a name
    model_info = mlflow.xgboost.log_model(xg, name = "best_params_xgboost_model")
    
    # get the model predicted values
    y_pred = model.predict(X_test)

    # get the probabilite values, so we can use for log loss and roc_auc
    y_proba = model.predict_proba(X_test)

    # get the models metrics
    score = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba[:, 1])
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])

    # log the metrics
    mlflow.log_metrics({"log_loss": ll, "accuracy_score": score, "roc_auc": roc_auc})

CPU times: user 1min 52s, sys: 2.73 s, total: 1min 55s
Wall time: 25 s
